# Canonical GHZ3 Bell baseline

This notebook prepares a deterministic `canonical_ez` direct-basis input for the three-qutrit GHZ Bell experiment. By default, Run All submits only the local Aer baseline; IQM submits a remote job only after `RUN_IQM` is set to `True`. Credentials remain provider/environment-only and are never embedded or persisted by this notebook.

In [ ]:
import hashlib
import json
import stat
import sys
from pathlib import Path
from uuid import uuid4

import numpy as np
import qiskit.qpy as qpy
from qiskit.quantum_info import Statevector


def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "qudits_on_qubits"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "Cannot find repository root. Start from this repository or a descendant "
        "containing pyproject.toml and src/qudits_on_qubits."
    )


def _checked_iqm_path(path, expected, description):
    path = Path(path)
    try:
        metadata = path.lstat()
    except FileNotFoundError:
        return None
    except OSError as error:
        raise RuntimeError(f"Cannot inspect {description}: {path}") from error
    reparse_attribute = getattr(stat, "FILE_ATTRIBUTE_REPARSE_POINT", 0x400)
    if stat.S_ISLNK(metadata.st_mode) or bool(
        getattr(metadata, "st_file_attributes", 0) & reparse_attribute
    ):
        raise RuntimeError(f"Refusing symlink or reparse {description}: {path}")
    if expected == "file" and not stat.S_ISREG(metadata.st_mode):
        raise RuntimeError(
            f"Malformed Git metadata or IQM .env candidate: expected file at {path}"
        )
    if expected == "dir" and not stat.S_ISDIR(metadata.st_mode):
        raise RuntimeError(f"Malformed Git metadata: expected directory at {path}")
    return path.resolve()


def resolve_iqm_env_path(repo_root):
    repo_root = Path(repo_root)
    local_env = _checked_iqm_path(
        repo_root / ".env", "file", "IQM .env candidate"
    )
    if local_env is not None:
        return local_env
    git_metadata = repo_root / ".git"
    metadata = _checked_iqm_path(git_metadata, "any", "Git metadata")
    if metadata is None:
        raise RuntimeError(
            f"Missing Git metadata; cannot resolve IQM .env for {repo_root}"
        )
    owning_repo = repo_root.resolve()
    if stat.S_ISREG(git_metadata.lstat().st_mode):
        try:
            gitdir_line = metadata.read_text(encoding="utf-8").strip()
        except OSError as error:
            raise RuntimeError(
                f"Malformed Git metadata: cannot read {git_metadata}"
            ) from error
        if not gitdir_line.lower().startswith("gitdir:"):
            raise RuntimeError(
                f"Malformed Git metadata: expected gitdir in {git_metadata}"
            )
        gitdir_value = gitdir_line[7:].strip()
        if not gitdir_value:
            raise RuntimeError(
                f"Malformed Git metadata: empty gitdir in {git_metadata}"
            )
        gitdir = Path(gitdir_value)
        if not gitdir.is_absolute():
            gitdir = metadata.parent / gitdir
        gitdir = _checked_iqm_path(gitdir, "dir", "Git worktree metadata")
        if gitdir is None:
            raise RuntimeError(
                f"Malformed Git metadata: missing worktree directory for {git_metadata}"
            )
        commondir = _checked_iqm_path(
            gitdir / "commondir", "file", "Git commondir metadata"
        )
        if commondir is None:
            raise RuntimeError(
                f"Malformed Git metadata: missing commondir in {gitdir}"
            )
        try:
            commondir_value = commondir.read_text(encoding="utf-8").strip()
        except OSError as error:
            raise RuntimeError(
                f"Malformed Git metadata: cannot read {commondir}"
            ) from error
        if not commondir_value:
            raise RuntimeError(
                f"Malformed Git metadata: empty commondir in {commondir}"
            )
        common_git_dir = Path(commondir_value)
        if not common_git_dir.is_absolute():
            common_git_dir = gitdir / common_git_dir
        common_git_dir = _checked_iqm_path(
            common_git_dir, "dir", "Git common directory"
        )
        if common_git_dir is None:
            raise RuntimeError(
                f"Malformed Git metadata: missing common directory for {gitdir}"
            )
        if common_git_dir.name != ".git":
            raise RuntimeError(
                "Cannot use non-.git common directory for IQM .env fallback; "
                "provide checkout-local .env or explicit env_path."
            )
        non_bare_error = (
            "Cannot validate non-bare owning repository for IQM .env fallback; "
            "provide checkout-local .env or explicit env_path."
        )
        try:
            owner_git_dir = _checked_iqm_path(
                common_git_dir.parent / ".git", "dir", "Git common directory"
            )
            expected_worktrees_dir = _checked_iqm_path(
                common_git_dir / "worktrees", "dir", "Git worktrees directory"
            )
        except (OSError, RuntimeError, ValueError):
            raise RuntimeError(non_bare_error) from None
        if (
            owner_git_dir != common_git_dir
            or expected_worktrees_dir is None
            or gitdir.parent != expected_worktrees_dir
        ):
            raise RuntimeError(non_bare_error)
        owning_repo = _checked_iqm_path(
            common_git_dir.parent, "dir", "owning repository"
        )
        if owning_repo is None:
            raise RuntimeError(
                "Cannot validate owning repository for IQM .env fallback; "
                "provide checkout-local .env or explicit env_path."
            )
        ownership_error = (
            "Cannot validate worktree ownership for IQM .env fallback; "
            "provide checkout-local .env or explicit env_path."
        )
        try:
            backlink = _checked_iqm_path(
                gitdir / "gitdir", "file", "Git worktree backlink"
            )
        except (OSError, RuntimeError):
            raise RuntimeError(ownership_error) from None
        if backlink is None:
            raise RuntimeError(ownership_error)
        try:
            backlink_value = backlink.read_text(encoding="utf-8").strip()
        except (OSError, UnicodeError):
            raise RuntimeError(ownership_error) from None
        if not backlink_value:
            raise RuntimeError(ownership_error)
        try:
            backlink_checkout = Path(backlink_value)
            if not backlink_checkout.is_absolute():
                backlink_checkout = gitdir / backlink_checkout
            backlink_checkout = backlink_checkout.resolve(strict=True)
        except (OSError, RuntimeError, ValueError):
            raise RuntimeError(ownership_error) from None
        if backlink_checkout != metadata:
            raise RuntimeError(ownership_error)
    candidates = [repo_root / ".env"]
    if owning_repo != repo_root.resolve():
        candidates.append(owning_repo / ".env")
    for candidate in candidates:
        checked = _checked_iqm_path(
            candidate, "file", "IQM .env candidate"
        )
        if checked is not None:
            return checked
    raise RuntimeError(
        f"Cannot find IQM .env file for checkout or owning repository: {repo_root}"
    )


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qudits_on_qubits.reference_experiments import get_encoding, get_reference_experiment
from qudits_on_qubits.benchmarks.direct_basis.circuits import build_direct_basis_graph_state_circuit
from qudits_on_qubits.experiments import (
    AerIdeal,
    BootstrapConfig,
    ExperimentSpec,
    IQMHardware,
    IQMQubitSelectorConfig,
    MitigationConfig,
    PathBasis,
    WorkloadOptimizationConfig,
    run_experiment,
)

In [ ]:
EXPECTED_STATE = "ghz3"
EXPECTED_QUBITS = 6


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def load_single_circuit(path):
    try:
        with Path(path).open("rb") as handle:
            circuits = qpy.load(handle)
    except Exception as error:
        raise RuntimeError(f"unable to load canonical basis QPY: {path}") from error
    if len(circuits) != 1:
        raise RuntimeError(
            f"canonical basis QPY must contain exactly one circuit, found {len(circuits)}"
        )
    return circuits[0]


def validate_canonical_basis(directory, expected_encoding, expected_circuit):
    directory = Path(directory)
    required_files = {"graph_state_direct_basis.qpy", "E.npy", "metadata.json"}
    try:
        actual_files = {path.name for path in directory.iterdir()}
    except OSError as error:
        raise RuntimeError(
            f"canonical basis directory is unavailable: {directory}"
        ) from error
    if actual_files != required_files:
        raise RuntimeError(
            "canonical basis files must be exactly graph_state_direct_basis.qpy, E.npy, "
            f"and metadata.json; found {sorted(actual_files)}"
        )

    encoding_path = directory / "E.npy"
    try:
        encoding = np.load(encoding_path, allow_pickle=False)
    except Exception as error:
        raise RuntimeError(
            f"canonical basis encoding is invalid: {encoding_path}"
        ) from error
    if encoding.shape != (4, 3):
        raise RuntimeError(
            f"canonical basis encoding must have shape (4, 3), got {encoding.shape}"
        )
    try:
        is_finite = np.isfinite(encoding).all()
    except TypeError as error:
        raise RuntimeError(
            "canonical basis encoding must be numeric and finite"
        ) from error
    if not is_finite:
        raise RuntimeError("canonical basis encoding must be finite")
    if not np.allclose(
        encoding.conj().T @ encoding, np.eye(3), atol=1e-12, rtol=0
    ):
        raise RuntimeError("canonical basis encoding must be an isometry")
    if not np.array_equal(encoding, expected_encoding):
        raise RuntimeError(
            "canonical basis encoding does not match canonical_ez"
        )

    circuit_path = directory / "graph_state_direct_basis.qpy"
    circuit = load_single_circuit(circuit_path)
    if circuit.num_qubits != EXPECTED_QUBITS or circuit.num_clbits != 0:
        raise RuntimeError(
            "canonical basis QPY must contain one unmeasured six-qubit circuit"
        )
    for instruction in circuit.data:
        operation = instruction.operation
        if operation.name in {"measure", "reset"}:
            raise RuntimeError(
                "canonical basis QPY must not contain measurements or resets"
            )
        if getattr(operation, "condition", None) is not None:
            raise RuntimeError(
                "canonical basis QPY must not contain conditioned instructions"
            )
        if getattr(operation, "blocks", ()):
            raise RuntimeError(
                "canonical basis QPY must not contain control flow"
            )
    try:
        same_state = Statevector.from_instruction(circuit).equiv(
            Statevector.from_instruction(expected_circuit)
        )
    except Exception as error:
        raise RuntimeError(
            "canonical basis QPY circuit cannot be validated as a state preparation"
        ) from error
    if not same_state:
        raise RuntimeError(
            "canonical basis QPY circuit does not match the canonical graph state"
        )

    metadata_path = directory / "metadata.json"
    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    except Exception as error:
        raise RuntimeError(
            f"canonical basis metadata is invalid: {metadata_path}"
        ) from error
    expected_metadata = {
        "schema": "qoq-reference-basis-v1",
        "state": EXPECTED_STATE,
        "encoding_id": "canonical_ez",
        "num_qubits": EXPECTED_QUBITS,
        "encoding_shape": [4, 3],
        "files": {
            "graph_state_direct_basis.qpy": {
                "sha256": sha256_file(circuit_path)
            },
            "E.npy": {"sha256": sha256_file(encoding_path)},
        },
    }
    if metadata != expected_metadata:
        raise RuntimeError(
            "canonical basis metadata does not match the validated bundle"
        )


def prepare_canonical_basis(repo_root):
    repo_root = Path(repo_root)
    expected_encoding = get_encoding("canonical_ez").as_array()
    expected_circuit = build_direct_basis_graph_state_circuit(
        EXPECTED_STATE, expected_encoding
    )
    parent = (
        repo_root / "experiment_inputs" / "reference_bases" / EXPECTED_STATE
    )
    directory = parent / "canonical_ez"

    if directory.exists():
        validate_canonical_basis(directory, expected_encoding, expected_circuit)
        return directory

    parent.mkdir(parents=True, exist_ok=True)
    staging_directory = parent / f".canonical_ez.tmp-{uuid4().hex}"
    staging_directory.mkdir()
    staging_files = (
        staging_directory / "graph_state_direct_basis.qpy",
        staging_directory / "E.npy",
        staging_directory / "metadata.json",
    )

    def cleanup_staging():
        for staging_file in staging_files:
            if staging_file.exists():
                staging_file.unlink()
        if staging_directory.exists():
            staging_directory.rmdir()

    try:
        qpy_path, encoding_path, metadata_path = staging_files
        with qpy_path.open("wb") as handle:
            qpy.dump(expected_circuit, handle)
        with encoding_path.open("wb") as handle:
            np.save(handle, expected_encoding, allow_pickle=False)
        metadata = {
            "schema": "qoq-reference-basis-v1",
            "state": EXPECTED_STATE,
            "encoding_id": "canonical_ez",
            "num_qubits": EXPECTED_QUBITS,
            "encoding_shape": [4, 3],
            "files": {
                "graph_state_direct_basis.qpy": {
                    "sha256": sha256_file(qpy_path)
                },
                "E.npy": {"sha256": sha256_file(encoding_path)},
            },
        }
        metadata_path.write_text(
            json.dumps(metadata, indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )

        validate_canonical_basis(
            staging_directory, expected_encoding, expected_circuit
        )
        try:
            staging_directory.rename(directory)
        except FileExistsError:
            cleanup_staging()
            validate_canonical_basis(
                directory, expected_encoding, expected_circuit
            )
        return directory
    finally:
        cleanup_staging()

In [ ]:
CANONICAL_BASIS_DIRECTORY = prepare_canonical_basis(REPO_ROOT)
CANONICAL_BASIS_DIRECTORY

## Shared configuration

The canonical reference, uncertainty settings, and hardware mitigation policy are shared across the Aer and IQM baselines. With `iqm_qubit_selector` enabled, the `initial_layouts` model field is retained for API compatibility, but each candidate is a canonically sorted physical routing subgraph, not an ordered logical-to-physical mapping.

In [ ]:
SHOTS = 100
IQM_ROUTING_SUBGRAPH_CANDIDATES = ((0, 1, 2, 3, 4, 7),)
IQM_SEED_CANDIDATES = (3, 7, 13)
IQM_LAYOUT_SELECTOR = IQMQubitSelectorConfig(
    top_k=10,
    num_trials=2000,
    cost_function="cz",
    readout_mode="none",
)
workload_optimization = WorkloadOptimizationConfig(
    initial_layouts=IQM_ROUTING_SUBGRAPH_CANDIDATES,
    seed_transpilers=IQM_SEED_CANDIDATES,
    iqm_qubit_selector=IQM_LAYOUT_SELECTOR,
)
UNCERTAINTY = BootstrapConfig(samples=2_000, seed=7)
HARDWARE_MITIGATION = MitigationConfig(
    readout=True,
    zne=True,
    zne_factors=(1, 3, 5),
    circuit_twirling=True,
    twirling_instances=5,
    twirling_seed=7,
)


def build_iqm_spec(
    basis_directory,
    *,
    env_path=None,
    output_root=Path("artifacts/experiment_runs"),
):
    return ExperimentSpec(
        state="ghz3",
        basis=PathBasis(basis_directory),
        backend=IQMHardware(
            device="garnet", use_metrics=True, env_path=env_path
        ),
        shots=SHOTS,
        mitigation=HARDWARE_MITIGATION,
        uncertainty=UNCERTAINTY,
        workload_optimization=workload_optimization,
        tags={"baseline": "canonical_ez", "backend": "iqm_garnet"},
        output_root=output_root,
    )


REFERENCE = get_reference_experiment("ghz3")
RESULTS = {}

## Aer ideal baseline

This unguarded local run records the canonical ideal-backend result.

In [ ]:
AER_SPEC = ExperimentSpec(
    state="ghz3",
    basis=PathBasis(CANONICAL_BASIS_DIRECTORY),
    backend=AerIdeal(seed_simulator=11),
    shots=SHOTS,
    uncertainty=UNCERTAINTY,
    tags={"baseline": "canonical_ez", "backend": "aer_ideal"},
)
AER_RESULT = run_experiment(AER_SPEC, repo_root=REPO_ROOT)
RESULTS["aer_ideal"] = AER_RESULT

## IQM Garnet baseline

Submission is opt-in; the default keeps this hardware run skipped.

In [ ]:
RUN_IQM = False

if RUN_IQM:
    IQM_ENV_PATH = resolve_iqm_env_path(REPO_ROOT)
    IQM_SPEC = build_iqm_spec(
        CANONICAL_BASIS_DIRECTORY, env_path=IQM_ENV_PATH
    )
    IQM_RESULT = run_experiment(IQM_SPEC, repo_root=REPO_ROOT)
    RESULTS["iqm_garnet"] = IQM_RESULT
else:
    print("IQM Garnet skipped; set RUN_IQM = True to submit.")

## Result summary

The summary reuses runner-produced values without recomputation, reads persisted workload-selection evidence, and compares the frozen classical bound only with an explicitly unconditional Bell estimate.

In [ ]:
UNCONDITIONAL_ESTIMATE_KEYS = (
    "zne_readout_mitigated_unconditional",
    "zne_unconditional",
    "readout_mitigated_unconditional",
    "raw_unconditional",
)


def load_workload_optimization(result):
    if result is None:
        return None
    try:
        document = json.loads(
            (Path(result.artifact_dir) / "experiment.json").read_text(
                encoding="utf-8"
            )
        )
    except (
        AttributeError,
        OSError,
        TypeError,
        UnicodeError,
        json.JSONDecodeError,
    ):
        return None
    workload_optimization = (
        document.get("workload_optimization")
        if isinstance(document, dict)
        else None
    )
    return (
        workload_optimization
        if isinstance(workload_optimization, dict)
        else None
    )


def select_unconditional_estimate(values):
    for key in UNCONDITIONAL_ESTIMATE_KEYS:
        estimate = values.get(key)
        if estimate is not None:
            return key, estimate
    return None, None


def estimate_real(estimate):
    if not isinstance(estimate, dict):
        return None
    value = estimate.get("estimate")
    if isinstance(value, dict):
        value = value.get("real")
    if isinstance(value, bool):
        return None
    try:
        value = float(value)
    except (TypeError, ValueError, OverflowError):
        return None
    return value if np.isfinite(value) else None


def summarize_results(results, reference):
    rows = []
    classical_bound = reference.bell_functional.classical_bound
    for backend, missing_status in (
        ("aer_ideal", "not_run"),
        ("iqm_garnet", "skipped"),
    ):
        result = results.get(backend)
        values = {} if result is None else result.values
        workload_optimization = load_workload_optimization(result)
        selected_workload = (
            workload_optimization.get("selected_workload")
            if workload_optimization is not None
            else None
        )
        selected_workload_aggregate = (
            selected_workload.get("aggregate")
            if isinstance(selected_workload, dict)
            else None
        )
        selected_unconditional_source, selected_unconditional = (
            select_unconditional_estimate(values)
        )
        selected_unconditional_real = estimate_real(selected_unconditional)
        rows.append(
            {
                "backend": backend,
                "status": (
                    missing_status if result is None else result.status.value
                ),
                "raw": values.get("raw"),
                "raw_conditional": values.get("raw_conditional"),
                "raw_unconditional": values.get("raw_unconditional"),
                "raw_invalid_codeword_rate": values.get(
                    "raw_invalid_codeword_rate"
                ),
                "readout_mitigated": values.get("readout_mitigated"),
                "readout_mitigated_conditional": values.get(
                    "readout_mitigated_conditional"
                ),
                "readout_mitigated_unconditional": values.get(
                    "readout_mitigated_unconditional"
                ),
                "readout_effective_invalid_codeword_weight": values.get(
                    "readout_effective_invalid_codeword_weight"
                ),
                "zne": values.get("zne"),
                "zne_conditional": values.get("zne_conditional"),
                "zne_unconditional": values.get("zne_unconditional"),
                "zne_readout_mitigated": values.get(
                    "zne_readout_mitigated"
                ),
                "zne_readout_mitigated_conditional": values.get(
                    "zne_readout_mitigated_conditional"
                ),
                "zne_readout_mitigated_unconditional": values.get(
                    "zne_readout_mitigated_unconditional"
                ),
                "diagnostics": values.get("diagnostics"),
                "leakage_rate": values.get("leakage_rate"),
                "workload_optimization": workload_optimization,
                "selected_layout": (
                    workload_optimization.get("selected_layout")
                    if workload_optimization is not None
                    else None
                ),
                "selected_seed_transpiler": (
                    workload_optimization.get("selected_seed_transpiler")
                    if workload_optimization is not None
                    else None
                ),
                "selected_workload_aggregate": selected_workload_aggregate,
                "selected_unconditional_source": selected_unconditional_source,
                "selected_unconditional": selected_unconditional,
                "classical_bound": classical_bound,
                "unconditional_exceeds_classical_bound": (
                    None
                    if selected_unconditional_real is None
                    else selected_unconditional_real > classical_bound
                ),
                "ideal_bell_value": reference.expected.ideal_bell_value,
                "artifact_dir": (
                    None if result is None else str(result.artifact_dir)
                ),
            }
        )
    return rows

In [ ]:
SUMMARY = summarize_results(RESULTS, REFERENCE)
SUMMARY